# 03b — ChemBERTa-2 baseline (full fine-tune)

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**Phase 3, second half.** Notebook 03a established the *classical* baseline (Random Forest on Morgan fingerprints). This notebook establishes the *pretrained-LLM* baseline.

**What ChemBERTa-2 is:** a RoBERTa-style transformer (~77M parameters) pretrained on millions of SMILES strings from ZINC and PubChem. It already "understands" chemical syntax. We fine-tune it (full fine-tune — all weights update) with a small task head per dataset.

**Why this matters for the FYP comparison:**
- LoRA/QLoRA in Phases 04/05 train *fewer* parameters than this full fine-tune
- We need the full-fine-tune number to say things like:
  *"LoRA matched ChemBERTa-2 within 2% AUC while updating only 0.5% of the parameters."*
- Without this row in the scoreboard, the LoRA/QLoRA claims are vague.

**What this notebook does:**
1. Loads the scaffold splits saved by Phase 02
2. Tokenises SMILES with ChemBERTa-2's tokenizer
3. Fine-tunes a fresh classification/regression head on BBBP, BACE, ESOL — 3 epochs each
4. Evaluates on the held-out test set
5. Appends 3 rows to `results/baselines.csv` (the shared scoreboard)

**Expected runtime on T4:** ~30–40 minutes total (~10–15 min per dataset).

## 1. Install + imports

`transformers`, `datasets`, `accelerate` for training; `scikit-learn` for metrics; `pandas` for the scoreboard.

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("torch         :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

## 2. Mount Drive + load Phase 02 splits

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR    = "/content/drive/MyDrive/chemistry-peft-fyp/data"
RESULTS_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/results"
RESULTS_CSV = f"{RESULTS_DIR}/baselines.csv"
os.makedirs(RESULTS_DIR, exist_ok=True)

def load_splits(name):
    prefix = name.lower()
    tr = pd.read_csv(f"{DATA_DIR}/{prefix}_train.csv")
    va = pd.read_csv(f"{DATA_DIR}/{prefix}_val.csv")
    te = pd.read_csv(f"{DATA_DIR}/{prefix}_test.csv")
    print(f"{name:5s}: train={len(tr)}  val={len(va)}  test={len(te)}")
    return tr, va, te

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}

## 3. Load ChemBERTa-2 tokenizer

**Model:** `DeepChem/ChemBERTa-77M-MTR` — the ~77M-parameter RoBERTa pretrained on SMILES with a multi-task regression objective. Standard "ChemBERTa-2" reference checkpoint.

We load the tokenizer once and reuse for all three datasets.

In [ ]:
MODEL_ID = "DeepChem/ChemBERTa-77M-MTR"
MAX_LEN  = 256  # 256 tokens covers >99% of SMILES in these datasets

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Sample tokenisation  :", tokenizer.tokenize("CC(=O)Oc1ccccc1C(=O)O")[:15], "...")

## 4. Helpers: tokenise + metrics

Two small helpers reused across all three datasets. Tokenisation converts each SMILES into the input IDs / attention mask that `Trainer` expects. Metrics are dataset-dependent (classification vs regression).

In [ ]:
def df_to_hf_dataset(df, is_regression):
    """Convert a pandas (smiles, label) DataFrame into a tokenised HF Dataset."""
    label_dtype = np.float32 if is_regression else np.int64
    ds = Dataset.from_dict({
        "smiles": df["smiles"].tolist(),
        "labels": df["label"].astype(label_dtype).tolist(),
    })

    def tok(batch):
        return tokenizer(batch["smiles"], truncation=True, padding=False, max_length=MAX_LEN)

    ds = ds.map(tok, batched=True, remove_columns=["smiles"])
    return ds

def make_metrics_fn(is_regression):
    def compute(eval_pred):
        logits, labels = eval_pred
        if is_regression:
            preds = logits.squeeze()
            rmse = float(np.sqrt(mean_squared_error(labels, preds)))
            mae  = float(mean_absolute_error(labels, preds))
            return {"rmse": rmse, "mae": mae}
        # classification
        proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        pred  = logits.argmax(axis=-1)
        return {
            "roc_auc": float(roc_auc_score(labels, proba)),
            "f1":      float(f1_score(labels, pred)),
        }
    return compute

## 5. Fine-tune ChemBERTa-2 on each dataset

For each dataset:
1. Load a *fresh* ChemBERTa-2 (so BACE doesn't inherit BBBP's task head)
2. Tokenise train + test
3. Fine-tune for 3 epochs
4. Evaluate on test

**Hyperparameters** (taken from the published ChemBERTa-2 fine-tune recipe):
- learning rate: 5e-5
- batch size: 16 (small enough for T4)
- 3 epochs
- weight decay: 0.01

These are deliberately default — like the RF baseline, the point is *standard* settings so the number is a fair baseline.

In [ ]:
results = []

def fine_tune_eval(name, is_regression):
    print(f"\n{'='*60}\n{name} ({'regression' if is_regression else 'classification'})\n{'='*60}")
    tr_df, _, te_df = splits[name]

    tr_ds = df_to_hf_dataset(tr_df, is_regression)
    te_ds = df_to_hf_dataset(te_df, is_regression)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=1 if is_regression else 2,
        problem_type="regression" if is_regression else "single_label_classification",
    )

    args = TrainingArguments(
        output_dir=f"/content/chemberta_{name.lower()}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="no",          # don't checkpoint — saves disk and time
        report_to="none",            # disable W&B / TensorBoard auto-logging
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=te_ds,
        processing_class=tokenizer,
        compute_metrics=make_metrics_fn(is_regression),
    )

    trainer.train()
    metrics = trainer.evaluate()
    print("\nFinal test metrics:", metrics)

    if is_regression:
        row = {
            "model": "ChemBERTa-2-full-FT", "dataset": name, "task": "regression",
            "metric_primary":   "rmse", "value_primary":   round(metrics["eval_rmse"], 4),
            "metric_secondary": "mae",  "value_secondary": round(metrics["eval_mae"], 4),
            "n_train": len(tr_ds), "n_test": len(te_ds),
        }
    else:
        row = {
            "model": "ChemBERTa-2-full-FT", "dataset": name, "task": "classification",
            "metric_primary":   "roc_auc", "value_primary":   round(metrics["eval_roc_auc"], 4),
            "metric_secondary": "f1",      "value_secondary": round(metrics["eval_f1"], 4),
            "n_train": len(tr_ds), "n_test": len(te_ds),
        }
    results.append(row)

    # Free GPU memory between runs
    del model, trainer
    torch.cuda.empty_cache()

fine_tune_eval("BBBP", is_regression=False)
fine_tune_eval("BACE", is_regression=False)
fine_tune_eval("ESOL", is_regression=True)

## 6. Append to the shared scoreboard

Same pattern as 03a — replace any prior `ChemBERTa-2-full-FT` rows so re-runs don't duplicate, and persist to Drive.

In [ ]:
new_rows = pd.DataFrame(results)

if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    existing = existing[existing["model"] != "ChemBERTa-2-full-FT"]
    combined = pd.concat([existing, new_rows], ignore_index=True)
else:
    combined = new_rows

combined.to_csv(RESULTS_CSV, index=False)
print(f"Wrote {len(new_rows)} rows to {RESULTS_CSV}\n")
print("Final baseline scoreboard:")
print(combined.to_string(index=False))

## 7. Done

**Phase 3 — fully complete:**
- ✅ Computed Morgan fingerprints (radius 2, 2048 bits)
- ✅ Trained Random Forest on BBBP/BACE/ESOL (03a)
- ✅ Loaded ChemBERTa-2 from Hugging Face (03b)
- ✅ Fine-tuned ChemBERTa-2 on BBBP, BACE, ESOL (03b)
- ✅ Evaluated ChemBERTa-2 on all three test sets (03b)
- ✅ Recorded all 6 numbers in `results/baselines.csv`
- ✅ Both baseline notebooks committed to GitHub

**Expected ChemBERTa-2 numbers (scaffold-split, rough literature range):**
- BBBP ROC-AUC ≈ 0.70–0.78
- BACE ROC-AUC ≈ 0.80–0.86
- ESOL RMSE   ≈ 0.85–1.10

**Next (Phase 04):** LoRA fine-tune of TinyLlama (or chosen base model). Targets to beat: the numbers in this scoreboard.